# Урок 09 - Шаблон проектирования метапознания


## Настройка

Этот блокнот демонстрирует шаблон проектирования метапознания с использованием Microsoft Agent Framework.

**Требования:**
- Развертывание Azure OpenAI, настроенное через переменные окружения
- Выполнена аутентификация Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Что такое метакогниция?

Метакогниция — это **мышление о мышлении**. В контексте агентов ИИ это означает создание агентов, которые могут:

- **Рефлексировать** над своими ответами и процессом рассуждения
- **Обнаруживать ошибки** и корректно восстанавливаться вместо тихого отказа
- **Оценивать**, являются ли их ответы полными и полезными
- **Адаптировать** свою стратегию, когда первоначальный подход не срабатывает (например, переходя на резервную систему)

Метакогнитивный агент не просто отвечает на вопросы — он контролирует свою собственную работу и вносит коррективы на ходу.


## Основные и резервные инструменты

Распространённый паттерн метакогниции — **резервная стратегия**. Агент сначала пробует основной инструмент; если он не срабатывает (например, ошибка 404), агент распознаёт сбой и прозрачно переключается на резервный инструмент.

Это отражает реальные системы, в которых основные сервисы могут быть недоступны, и агентам необходимо самостоятельно диагностировать проблему, прежде чем выбрать альтернативный путь.

Ниже мы определяем два инструмента для поиска рейсов:
- **Основной** — охватывает Париж, Токио и Барселону
- **Резервный** — охватывает Берлин, Сидней и Нью-Йорк


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Саморефлексирующий агент с восстановлением после ошибок

Нижеприведённому агенту поручено сначала воспользоваться основной системой полёта, распознавать отказы и прозрачно переключаться на резервную систему. После каждого ответа он кратко самооценит, полностью ли он ответил на вопрос пользователя.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Шаблон самооценки

Еще одна сторона метакогниции — **самооценка**: отдельный агент (или тот же агент при втором проходе) проверяет ответ на полноту, точность и полезность.

Ниже мы создаём агента `ResponseEvaluator`, который оценивает ответы агента по путешествиям по трём параметрам.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Резюме

В этом уроке вы узнали, как создавать **метакогнитивные агенты** с помощью Microsoft Agent Framework:

- **Саморефлексия**: Агенты, которые отслеживают собственное рассуждение и прозрачно сообщают о том, что произошло.
- **Восстановление после ошибок с резервными вариантами**: Шаблон «основной + резервный инструмент», в котором агент обнаруживает сбои (например, ошибки 404) и автоматически пытается обратиться к альтернативному источнику.
- **Самооценка**: Отдельный агент-оценщик, который оценивает ответы по полноте, точности и полезности.

Эти шаблоны делают агентов более устойчивыми, прозрачными и заслуживающими доверия — критически важные качества для промышленного использования.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Отказ от ответственности:
Этот документ был переведён с помощью сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на стремление к точности, пожалуйста, имейте в виду, что автоматические переводы могут содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для критически важной информации рекомендуется профессиональный перевод, выполненный человеком. Мы не несем ответственности за любые недоразумения или неверные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
